# Week 2 — Residual Stream & Information Flow

> Hooking the residual stream of a transformer to attribute information flow per-head and per-MLP.

---

## 1. Theory

### 1.1 The residual stream view

A transformer with $L$ layers, each with $H$ attention heads and an MLP, can be written as

$$x_{\ell+1} \;=\; x_\ell \;+\; \sum_{h=1}^H A^{(\ell, h)}(x_\ell) \;+\; M^{(\ell)}(x_\ell)$$

The **residual stream** $x_\ell$ is the shared bus into which every head and MLP writes its rank-bounded contribution. Elhage et al. (2021) argued that this *additive* structure makes the transformer fundamentally interpretable through circuit analysis on the stream, not on attention patterns alone.

### 1.2 Why attention heatmaps are misleading

Attention weights $\alpha^{(\ell,h)}_{ij}$ describe **where** information moves; they say nothing about **what** information moves nor how much. A head can have a perfectly diagonal attention pattern but write nothing meaningful — the value projection $V^{(\ell,h)} x_j$ may be near-zero.

The correct diagnostic is the **L2 norm of each head's write**:

$$\Big\|A^{(\ell,h)}(x_\ell)\Big\|_2 \;=\; \Big\|W_O^{(\ell,h)}\!\left(\sum_j \alpha^{(\ell,h)}_{ij}\, W_V^{(\ell,h)} x_j\right)\Big\|_2$$

### 1.3 Attention entropy & induction circuits

The Shannon entropy of attention row $\alpha^{(\ell,h)}_i$ classifies head behavior:

$$\mathcal{H}^{(\ell,h)}_i = -\sum_j \alpha^{(\ell,h)}_{ij}\,\log \alpha^{(\ell,h)}_{ij}$$

* $\mathcal{H} \to 0$ : a "copy" or induction head.
* $\mathcal{H} \approx \log T$ : a near-uniform averaging head.
* Heads where $\alpha_{:,0} \to 1$ (a column hot at the BOS position) are **attention sinks** — they neutralize the head's contribution and are common in language models trained with softmax attention (Xiao et al., 2023).

### 1.4 Induction score (Olsson et al., 2022)

Looks for the prefix-match-and-copy pattern:

$$\mathrm{IndScore}^{(h)} = \frac{1}{|\mathcal{I}|}\sum_{i \in \mathcal{I}} \alpha^{(h)}_{i, j(i) + 1}$$

where $j(i)$ is the most recent earlier occurrence of token $t_i$. Induction heads typically have $\mathrm{IndScore} \gtrsim 0.5$.

### 1.5 Information bottleneck per layer

In the residual stream framework, the per-layer mutual information $I(X_\ell; Y)$ (Tishby et al., 2000) is bounded by the sum of head writes:

$$I(X_{\ell+1}; Y) \;\le\; I(X_\ell; Y) + \sum_h I\bigl(A^{(\ell,h)}(X_\ell); Y\bigr)$$

— each head can at most contribute the information it writes.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.utils import set_global_seed
from src.utils.synthetic import AttentionConfig, synth_attention_matrix
from src.mechanistic import (
    ResidualStreamCapture,
    attention_entropy,
    classify_head_archetypes,
    induction_score,
)

set_global_seed(0)
print("TensorLens — Week 2 notebook loaded")


## 2. Building a minimal transformer for instrumentation

We build a small 4-layer / 4-head transformer with a residual stream of dimension $d=64$. Real LLMs use the same structure — only the dimensions change.

In [ ]:
class HeadOutput(nn.Module):
    """Wrap one attention head's output so we can hook it."""
    def __init__(self, d_head: int, d_model: int):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_head, bias=False)
        self.W_k = nn.Linear(d_model, d_head, bias=False)
        self.W_v = nn.Linear(d_model, d_head, bias=False)
        self.W_o = nn.Linear(d_head, d_model, bias=False)
        self.scale = d_head ** -0.5
        self.attn_weights: torch.Tensor | None = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        q, k, v = self.W_q(x), self.W_k(x), self.W_v(x)
        T = x.shape[-2]
        scores = (q @ k.transpose(-1, -2)) * self.scale
        causal = torch.tril(torch.ones(T, T, device=x.device)).bool()
        scores = scores.masked_fill(~causal, float("-inf"))
        attn = scores.softmax(dim=-1)
        self.attn_weights = attn.detach()
        out = self.W_o(attn @ v)
        return out


class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int):
        super().__init__()
        d_head = d_model // n_heads
        self.heads = nn.ModuleList([HeadOutput(d_head, d_model) for _ in range(n_heads)])
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_norm = self.ln1(x)
        for head in self.heads:
            x = x + head(x_norm)
        x = x + self.mlp(self.ln2(x))
        return x


class TinyTransformer(nn.Module):
    def __init__(self, vocab: int, d_model: int, n_layers: int, n_heads: int, d_ff: int):
        super().__init__()
        self.embed = nn.Embedding(vocab, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)
        ])

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        x = self.embed(ids)
        for blk in self.blocks:
            x = blk(x)
        return x


VOCAB = 32
D_MODEL = 64
N_LAYERS = 4
N_HEADS = 4
SEQ_LEN = 24

model = TinyTransformer(VOCAB, D_MODEL, N_LAYERS, N_HEADS, d_ff=2 * D_MODEL)
torch.manual_seed(1)
input_ids = torch.randint(0, VOCAB, (1, SEQ_LEN))
print(f"Model params: {sum(p.numel() for p in model.parameters())}")
print(f"Input ids shape: {input_ids.shape}")


## 3. Installing residual-stream hooks

`ResidualStreamCapture` walks the module tree, asks our selectors to identify (layer, head) targets, and installs forward hooks. Within the `with` block we run the forward pass — the captured record then exposes per-head writes for analysis.


In [ ]:
def head_selector(module: nn.Module, name: str):
    # Match attention heads by name pattern "blocks.{L}.heads.{H}"
    parts = name.split(".")
    if len(parts) == 4 and parts[0] == "blocks" and parts[2] == "heads":
        return int(parts[1]), int(parts[3])
    return None

def mlp_selector(module: nn.Module, name: str):
    # Match the MLP Sequential as a whole at "blocks.{L}.mlp"
    parts = name.split(".")
    if len(parts) == 3 and parts[0] == "blocks" and parts[2] == "mlp":
        return int(parts[1])
    return None


capture = ResidualStreamCapture(
    model,
    attention_head_selector=head_selector,
    mlp_selector=mlp_selector,
)

with capture:
    _ = model(input_ids)

print(capture.record.summary())
print("Attention write tensor shape (sample):", next(iter(capture.record.attention_writes.values())).shape)


## 4. Per-head write norms — the true contribution signal

Each cell here is the L2 norm of one attention head's per-token write to the residual stream, summed over tokens. This is the diagnostic Elhage et al. recommend instead of raw attention weights.

In [ ]:
norms = capture.record.attention_write_norms()  # (L, H) -> tensor (batch,)
norm_matrix = np.zeros((N_LAYERS, N_HEADS))
for (l, h), v in norms.items():
    norm_matrix[l, h] = float(v.mean())

mlp_norms = capture.record.mlp_write_norms()
mlp_vec = np.array([float(mlp_norms[l].mean()) for l in range(N_LAYERS)])

print("Attention write norms (rows=layer, cols=head):")
print(np.round(norm_matrix, 3))
print("\nMLP write norms (per layer):")
print(np.round(mlp_vec, 3))


In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.65, 0.35],
    subplot_titles=("Per-head residual-stream write norm", "Per-layer MLP write norm"),
)

fig.add_trace(
    go.Heatmap(
        z=norm_matrix,
        x=[f"H{h}" for h in range(N_HEADS)],
        y=[f"L{l}" for l in range(N_LAYERS)],
        colorscale="Viridis",
        colorbar=dict(title="‖A(x)‖₂", x=0.55, len=0.9),
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Bar(
        x=[f"L{l}" for l in range(N_LAYERS)],
        y=mlp_vec,
        marker_color="#7F3FFF",
        showlegend=False,
    ),
    row=1, col=2,
)

fig.update_layout(height=420, width=1100, title="Residual stream contributions")
fig.show()


## 5. Attention entropy & induction scores

We sweep the attention patterns of each layer's heads through `attention_entropy` and `induction_score`, then classify with `classify_head_archetypes`.

In [ ]:
# Stack each layer's attention matrices into (H, T, T)
layer_attn = {}
for l, blk in enumerate(model.blocks):
    layer_attn[l] = torch.stack(
        [head.attn_weights[0] for head in blk.heads], dim=0
    )

entropies = {}
archetypes = {}
ind_scores = {}
toks = input_ids[0]
for l, attn in layer_attn.items():
    ent = attention_entropy(attn)              # (H, T)
    entropies[l] = ent.mean(dim=-1).numpy()    # (H,)
    archetypes[l] = classify_head_archetypes(attn)
    ind_scores[l] = induction_score(attn, toks).numpy()

print("Per-layer mean head entropy:")
for l in range(N_LAYERS):
    print(f"  L{l}: entropy={np.round(entropies[l], 3).tolist()}  archetypes={archetypes[l]}  ind={np.round(ind_scores[l], 3).tolist()}")


In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Mean attention entropy per head", "Induction score per head"))
for l in range(N_LAYERS):
    fig.add_trace(
        go.Bar(name=f"L{l}", x=[f"H{h}" for h in range(N_HEADS)], y=entropies[l], showlegend=(l == 0)),
        row=1, col=1,
    )
    fig.add_trace(
        go.Bar(name=f"L{l}", x=[f"H{h}" for h in range(N_HEADS)], y=ind_scores[l], showlegend=False),
        row=1, col=2,
    )
fig.update_layout(height=420, width=1100, barmode="group", title="Head behavioral fingerprints")
fig.show()


## 6. Information-flow Sankey diagram

We weight edges by the *L2 norm of each head's write* — not by attention weights. Nodes are organized as columns: input → layer-0 heads → layer-1 heads → … → output. Edge weight = norm of (layer, head)'s write.

In [ ]:
# Build a layered Sankey: input → (L0 heads) → (L1 heads) → ... → output
labels = ["input"]
for l in range(N_LAYERS):
    for h in range(N_HEADS):
        labels.append(f"L{l}.H{h}")
labels.append("output")

node_index: dict[str, int] = {name: i for i, name in enumerate(labels)}

src, tgt, val = [], [], []
# input → L0 heads (uniform unit weight)
for h in range(N_HEADS):
    src.append(node_index["input"])
    tgt.append(node_index[f"L0.H{h}"])
    val.append(1.0)

# Within-layer heads to next-layer heads, weighted by next layer's head write norm
for l in range(N_LAYERS - 1):
    for h_curr in range(N_HEADS):
        for h_next in range(N_HEADS):
            src.append(node_index[f"L{l}.H{h_curr}"])
            tgt.append(node_index[f"L{l + 1}.H{h_next}"])
            val.append(float(norm_matrix[l + 1, h_next]) / N_HEADS)

# Last layer heads → output, weighted by their write norm
for h in range(N_HEADS):
    src.append(node_index[f"L{N_LAYERS - 1}.H{h}"])
    tgt.append(node_index["output"])
    val.append(float(norm_matrix[N_LAYERS - 1, h]))

fig = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(label=labels, pad=15, thickness=18),
    link=dict(source=src, target=tgt, value=val),
))
fig.update_layout(height=520, width=1100,
                  title="Information flow across the residual stream (edge width = per-head write norm)")
fig.show()


## 7. Take-aways

1. **Attention weights ≠ information flow.** A head with a sharp attention pattern can write a near-zero contribution; the norm of $W_O \cdot (\alpha V x)$ is the right metric.
2. **Archetype distribution is informative.** Even at random initialization, our 4×4 = 16 heads spread across `induction`, `sink`, `uniform`, and `mixed` classes; trained models concentrate at the corners.
3. **The Sankey is the bus map.** Plotting per-head write norm as Sankey edge width gives an at-a-glance picture of *which* heads dominate the residual stream — the natural starting point for any circuit analysis.

### References

* Elhage, N. et al. (2021). *A Mathematical Framework for Transformer Circuits.* Anthropic.
* Olsson, C. et al. (2022). *In-context Learning and Induction Heads.* Anthropic.
* Xiao, G. et al. (2023). *Efficient Streaming Language Models with Attention Sinks.* arXiv:2309.17453.
* Tishby, N., Pereira, F.C., Bialek, W. (2000). *The Information Bottleneck Method.*
